# Imports

In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from rdkit import Chem

PROJECT_ROOT = Path.cwd().parent  # si el notebook esta en /notebooks
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths_and_constants import Paths
_P = Paths()

# Functions

# Propertyprep for binding (biolip)

In [ ]:
biol = pd.read_csv(_P.data_processed_datasets / 'biolip_binding' / 'biolip_cleaned_with_negatives.csv', index_col=0)
print(biol.columns)

biol["protein_sequence"] = biol["protein_sequence"].astype(str).str.strip()
biol["uniprot_id"] = biol["uniprot_id"].replace("", pd.NA)

seq_to_uniprot = (
    biol.dropna(subset=["uniprot_id"])
      .drop_duplicates(subset=["protein_sequence"])
      .set_index("protein_sequence")["uniprot_id"]
)

biol["uniprot_id"] = biol["uniprot_id"].fillna(biol["protein_sequence"].map(seq_to_uniprot))

Index(['chain_id', 'resolution', 'binding_site_id', 'ligand_id',
       'ligand_chain_id', 'ligand_serial_number', 'binding_site_residues',
       'nearby_residues', 'ligand_contact_residues', 'catalytic_site_residues',
       'ec_number', 'go_terms', 'binding_affinity_1', 'binding_affinity_2',
       'binding_affinity_3', 'binding_affinity_4', 'uniprot_id', 'pubmed_id',
       'binding_site_range', 'protein_sequence', 'ligand_id_raw', 'smiles',
       'ligand_name', 'map_status', 'label'],
      dtype='object')


/tmp/ipykernel_12390/153839608.py:1: DtypeWarning: Columns (13,14,16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  biol = pd.read_csv(_P.data_processed_datasets / 'biolip_binding' / 'biolip_cleaned_with_negatives.csv', index_col=0)


In [ ]:
biol['smiles'] = biol['smiles'].astype(str).str.strip()
biol['ligand_id_raw'] = biol['ligand_id_raw'].replace("", pd.NA)
biol['ligand_name'] = biol['ligand_name'].replace("", pd.NA)

smiles_to_id = (
    biol.dropna(subset=['ligand_id_raw'])
      .drop_duplicates(subset=['smiles'])
      .set_index('smiles')['ligand_id_raw']
)

smiles_to_name = (
    biol.dropna(subset=['ligand_name'])
      .drop_duplicates(subset=['smiles'])
      .set_index('smiles')['ligand_name']
)

biol['ligand_id_raw'] = biol['ligand_id_raw'].fillna(biol['smiles'].map(smiles_to_id))
biol['ligand_name'] = biol['ligand_name'].fillna(biol['smiles'].map(smiles_to_name))

In [19]:
final_cols = ['uniprot_id', 'protein_sequence', 'ligand_id_raw', 'ligand_name', 'smiles', 'label']
biol_final = biol[final_cols]
biol_final = biol_final.reset_index(drop=True)
biol_final.to_csv(_P.data_processed_properties / 'binding' / 'biolip_binding.csv', index=False)

# Propertyprep for DDG (FireProtDB)

In [27]:
ddg = pd.read_csv(_P.data_processed_properties / 'ddg_mut' / 'fireprotdb_ddg.csv', index_col=0)
ddg = ddg.reset_index()
print(ddg.columns)

Index(['EXPERIMENT_ID', 'SEQUENCE_ID', 'MUTANT_ID', 'SOURCE_SEQUENCE_ID',
       'TARGET_SEQUENCE_ID', 'SEQUENCE_LENGTH', 'SUBSTITUTION', 'INSERTION',
       'DELETION', 'PROTEIN', 'ORGANISM', 'CHYMOTRYPSIN_ML', 'CM', 'DCP',
       'DDG', 'DG', 'DH', 'DHVH', 'DOMAINOME_DDG', 'DOMAINOME_DDG_STD',
       'DOMAINOME_FITNESS', 'DOMAINOME_FITNESS_STD', 'DTM', 'M',
       'REVERSIBILITY', 'STABILIZING', 'STATE', 'TM', 'TRYPSIN_ML', 'BUFFER',
       'BUFFER_CONC', 'EXP_TEMPERATURE', 'ION', 'ION_CONC', 'MEASURE',
       'METHOD', 'PH', 'EC_NUMBER', 'UNIPROTKB', 'INTERPRO', 'MEGASCALE',
       'ACTIVE_SITE', 'BINDING_SITE', 'CONSERVATION', 'WWPDB', 'B_FACTOR',
       'IN_TUNNEL', 'IN_POCKET', 'PUBLICATION_PMID', 'PUBLICATION_DOI',
       'PUBLICATION_YEAR', 'SOURCE_DATASET', 'REFERENCING_DATASET'],
      dtype='object')


In [ ]:
fire_cols = ['EXPERIMENT_ID', 'UNIPROTKB', 'mutation', 'ddg', 'label']